# 01 — Architecture Overview: What's Inside an LLM?

This notebook loads **Qwen 2.5 0.5B** and inspects every component of the model.

By the end, you'll know:
- Every module in a modern transformer and its exact shape
- Where the parameters actually live (spoiler: mostly in the MLP)
- How GQA, SwiGLU, RMSNorm, and RoPE work at the code level

**Requirements**: ~2GB RAM. No GPU needed.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import matplotlib.pyplot as plt
import numpy as np
from transformer_lens import HookedTransformer
from utils.model_loading import load_tlens_model, format_param_count
from utils.visualization import apply_theme, ACCENT_BLUE, ACCENT_ORANGE, ACCENT_GREEN, ACCENT_RED, ACCENT_PURPLE, ACCENT_TEAL, TEXT_COLOR, DARK_BG, DARK_SURFACE, DARK_GRID, PALETTE, plot_pie, _style_box

apply_theme()

# Change to "7b" if you have >= 16GB VRAM
MODEL_SIZE = "0.5b"
model = load_tlens_model(MODEL_SIZE)

## Model Configuration

Let's see the key architectural parameters.

In [ ]:
cfg = model.cfg
print(f"Model:              {cfg.model_name}")
print(f"Layers:             {cfg.n_layers}")
print(f"Hidden dimension:   {cfg.d_model}")
print(f"Head dimension:     {cfg.d_head}")
print(f"Query heads:        {cfg.n_heads}")
print(f"KV heads:           {cfg.n_key_value_heads}")
print(f"GQA ratio:          {cfg.n_heads // cfg.n_key_value_heads}:1 (queries per KV head)")
print(f"MLP dimension:      {cfg.d_mlp}")
print(f"Vocab size:         {cfg.d_vocab:,}")
print(f"Context length:     {cfg.n_ctx:,}")
print(f"Activation fn:      {cfg.act_fn}")
print(f"Normalization:      {'RMSNorm' if 'RMS' in str(cfg.normalization_type) else cfg.normalization_type}")
print(f"Positional encoding: RoPE (rotary)")
print(f"Total parameters:   {format_param_count(sum(p.numel() for p in model.parameters()))}")

## Module Tree: Every Weight Tensor in the Model

Every parameter in the model is a tensor of floating-point numbers. Let's enumerate all of them.

In [ ]:
# Enumerate every parameter
print(f"{'Parameter Name':<50} {'Shape':<25} {'Params':>12}")
print("=" * 90)

total = 0
for name, param in model.named_parameters():
    n = param.numel()
    total += n
    # Only print unique layers (layer 0) + shared params to avoid noise
    if any(x in name for x in ["blocks.0.", "embed", "unembed", "ln_final"]):
        print(f"{name:<50} {str(list(param.shape)):<25} {format_param_count(n):>12}")
    elif "blocks.1." in name and "blocks.0." not in name:
        print(f"  ... (layers 1-{cfg.n_layers - 1} identical)")
        # Only print this once
        if "ln1.w" in name:
            continue

print(f"\n{'TOTAL':<50} {'':<25} {format_param_count(total):>12}")

## Parameter Budget: Where Do the Parameters Live?

Let's break down the total parameter count by component type.

In [ ]:
# Count parameters by component type
counts = {"Embedding": 0, "Attention": 0, "MLP": 0, "Norm": 0, "Unembedding": 0}

for name, param in model.named_parameters():
    n = param.numel()
    if "embed" in name and "unembed" not in name:
        counts["Embedding"] += n
    elif "unembed" in name:
        counts["Unembedding"] += n
    elif "attn" in name:
        counts["Attention"] += n
    elif "mlp" in name:
        counts["MLP"] += n
    elif "ln" in name:
        counts["Norm"] += n

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Donut pie chart
pie_colors = [ACCENT_BLUE, ACCENT_ORANGE, ACCENT_GREEN, ACCENT_RED, ACCENT_PURPLE]
labels = [f"{k}\n{format_param_count(v)}" for k, v in counts.items()]
plot_pie(labels, list(counts.values()), colors=pie_colors, title="Parameter Budget by Component", ax=ax1)

# Stacked bar chart per layer
layer_attn, layer_mlp, layer_norm = [], [], []
for i in range(cfg.n_layers):
    layer_attn.append(sum(p.numel() for n, p in model.named_parameters() if f"blocks.{i}.attn" in n))
    layer_mlp.append(sum(p.numel() for n, p in model.named_parameters() if f"blocks.{i}.mlp" in n))
    layer_norm.append(sum(p.numel() for n, p in model.named_parameters() if f"blocks.{i}.ln" in n))

x = range(cfg.n_layers)
ax2.bar(x, layer_mlp, label="MLP", color=ACCENT_GREEN, alpha=0.85, edgecolor="none")
ax2.bar(x, layer_attn, bottom=layer_mlp, label="Attention", color=ACCENT_ORANGE, alpha=0.85, edgecolor="none")
ax2.bar(x, layer_norm, bottom=[a + m for a, m in zip(layer_attn, layer_mlp)],
        label="Norm", color=ACCENT_RED, alpha=0.85, edgecolor="none")
ax2.set_xlabel("Layer")
ax2.set_ylabel("Parameters")
ax2.set_title("Parameters per Layer")
ax2.legend()
plt.tight_layout()
plt.show()

print(f"\nMLP is {counts['MLP'] / sum(counts.values()) * 100:.1f}% of all parameters.")
print(f"Each transformer block has {format_param_count(layer_attn[0] + layer_mlp[0] + layer_norm[0])} params.")

## Grouped Query Attention (GQA)

Standard multi-head attention gives each head its own Q, K, and V projections. GQA saves memory by **sharing K and V projections** across groups of query heads.

```
Standard MHA:  Q0 K0 V0 | Q1 K1 V1 | Q2 K2 V2 | ... (every head has unique KV)
GQA:           Q0 Q1 Q2 Q3 Q4 Q5 Q6  →  K0 V0    (7 query heads share 1 KV head)
               Q7 Q8 Q9 Q10 Q11 Q12 Q13  →  K1 V1  (7 query heads share 1 KV head)
```

This reduces KV-cache memory by the GQA ratio (7x for 0.5B) with minimal quality loss.

In [ ]:
# GQA: the actual weight shapes tell the story
W_Q = model.blocks[0].attn.W_Q  # Query projection
W_K = model.blocks[0].attn.W_K  # Key projection
W_V = model.blocks[0].attn.W_V  # Value projection
W_O = model.blocks[0].attn.W_O  # Output projection

print("Attention weight shapes (layer 0):")
print(f"  W_Q: {list(W_Q.shape):>30}  →  {cfg.n_heads} query heads × {cfg.d_head} dim")
print(f"  W_K: {list(W_K.shape):>30}  →  {cfg.n_key_value_heads} KV heads × {cfg.d_head} dim")
print(f"  W_V: {list(W_V.shape):>30}  →  {cfg.n_key_value_heads} KV heads × {cfg.d_head} dim")
print(f"  W_O: {list(W_O.shape):>30}  →  recombines all heads back to d_model")

q_params = W_Q.numel()
kv_params = W_K.numel() + W_V.numel()
print(f"\n  Q params:  {format_param_count(q_params)}")
print(f"  KV params: {format_param_count(kv_params)}")
print(f"  If this were full MHA (each head with own KV): KV would be {format_param_count(kv_params * cfg.n_heads // cfg.n_key_value_heads)}")
print(f"  GQA saves {(1 - cfg.n_key_value_heads / cfg.n_heads) * 100:.0f}% of KV parameters and cache memory")

## SwiGLU: The Gated Feed-Forward Network

Instead of a simple `ReLU(x @ W1) @ W2`, modern LLMs use **SwiGLU** — a gated linear unit with SiLU (Swish) activation:

```
output = (SiLU(x @ W_gate) * (x @ W_up)) @ W_down
```

Three weight matrices instead of two. The gate learns which "neurons" to activate for each input.

In [ ]:
# SwiGLU MLP structure
mlp = model.blocks[0].mlp
W_gate = mlp.W_gate
W_in = mlp.W_in
W_out = mlp.W_out

print("MLP weight shapes (layer 0):")
print(f"  W_gate (gate projection):  {list(W_gate.shape):>20}  ->  d_model -> d_mlp")
print(f"  W_in   (up projection):    {list(W_in.shape):>20}  ->  d_model -> d_mlp")
print(f"  W_out  (down projection):  {list(W_out.shape):>20}  ->  d_mlp -> d_model")
print(f"\n  Expansion ratio: {cfg.d_mlp / cfg.d_model:.1f}x ({cfg.d_model} -> {cfg.d_mlp} -> {cfg.d_model})")
print(f"  MLP params per layer: {format_param_count(W_gate.numel() + W_in.numel() + W_out.numel())}")

# Visualize SiLU activation function
fig, ax = plt.subplots(figsize=(9, 4.5))
x = torch.linspace(-5, 5, 300)
silu = x * torch.sigmoid(x)
relu = torch.relu(x)
ax.plot(x.numpy(), silu.numpy(), label="SiLU (Swish)", linewidth=2.5, color=ACCENT_GREEN)
ax.plot(x.numpy(), relu.numpy(), label="ReLU", linewidth=2, color=ACCENT_RED, linestyle="--", alpha=0.5)
ax.fill_between(x.numpy(), silu.numpy(), alpha=0.08, color=ACCENT_GREEN)
ax.axhline(0, color=TEXT_COLOR, linewidth=0.4, alpha=0.3)
ax.axvline(0, color=TEXT_COLOR, linewidth=0.4, alpha=0.3)
ax.set_title("SiLU vs ReLU: The Activation Function in SwiGLU")
ax.set_xlabel("Input")
ax.set_ylabel("Output")
ax.legend()
plt.tight_layout()
plt.show()
print("\nSiLU is smooth (differentiable everywhere) and allows small negative values through.")

## RMSNorm: The Normalization Layer

Modern LLMs use **RMSNorm** instead of LayerNorm. It's simpler — no mean centering, no bias:

```
RMSNorm(x) = x / sqrt(mean(x²) + ε) × γ
```

Where `γ` is a learned scale parameter (initialized to 1.0).

In [ ]:
# RMSNorm: inspect the learned scale parameters
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

norm_colors = [ACCENT_BLUE, ACCENT_ORANGE, ACCENT_RED]
for idx, (name, title) in enumerate([
    ("blocks.0.ln1.w", "Layer 0: Pre-Attention Norm"),
    ("blocks.0.ln2.w", "Layer 0: Pre-MLP Norm"),
    ("ln_final.w", "Final Norm"),
]):
    param = dict(model.named_parameters())[name]
    data = param.detach().float().cpu().numpy()
    axes[idx].plot(data, linewidth=0.6, color=norm_colors[idx], alpha=0.9)
    axes[idx].axhline(1.0, color=TEXT_COLOR, linestyle="--", alpha=0.25, linewidth=0.8, label="init (1.0)")
    axes[idx].set_title(title)
    axes[idx].set_xlabel("Dimension")
    axes[idx].set_ylabel("$\\gamma$ (scale)")
    axes[idx].legend(fontsize=8)
    _style_box(axes[idx], f"$\\mu$={data.mean():.3f}\n$\\sigma$={data.std():.3f}", y=0.15, va="bottom")

plt.tight_layout()
plt.show()
print("After training, the scale parameters deviate from 1.0 — the model learned to amplify or dampen specific dimensions.")

## RoPE: Rotary Position Embeddings

Unlike older models that *add* position vectors to token embeddings, RoPE *rotates* the Q and K vectors by angles proportional to position. The key property: the dot product between rotated Q and K depends only on **relative position**, not absolute.

```
For dimension pair (2i, 2i+1) at position m:
    θᵢ = base^(-2i/d)       where base = 1,000,000
    q_rotated[2i]   = q[2i]·cos(m·θᵢ) - q[2i+1]·sin(m·θᵢ)
    q_rotated[2i+1] = q[2i]·sin(m·θᵢ) + q[2i+1]·cos(m·θᵢ)
```

In [ ]:
# Visualize RoPE frequencies
rope_base = 1_000_000  # Qwen 2.5 uses theta=1M
d_head = cfg.d_head
dim_indices = torch.arange(0, d_head, 2).float()
freqs = 1.0 / (rope_base ** (dim_indices / d_head))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Frequencies per dimension pair
ax1.plot(freqs.numpy(), marker=".", markersize=5, color=ACCENT_BLUE, linewidth=1.5)
ax1.fill_between(range(len(freqs)), freqs.numpy(), alpha=0.1, color=ACCENT_BLUE)
ax1.set_xlabel("Dimension Pair Index")
ax1.set_ylabel("Frequency (radians/position)")
ax1.set_yscale("log")
ax1.set_title(f"RoPE Frequencies (base={rope_base:,})")

# Rotation angles at different positions
positions = [0, 1, 10, 100, 1000]
for i, pos in enumerate(positions):
    angles = pos * freqs.numpy()
    ax2.plot(angles % (2 * np.pi), label=f"pos={pos}", alpha=0.85, linewidth=1.5, color=PALETTE[i])
ax2.set_xlabel("Dimension Pair Index")
ax2.set_ylabel("Rotation Angle (radians)")
ax2.set_title("Rotation Angles at Different Positions")
ax2.legend()

plt.tight_layout()
plt.show()

print("Low-indexed dimensions rotate fast (high frequency) -> capture nearby positions.")
print("High-indexed dimensions rotate slowly -> capture long-range relationships.")

## The Complete Data Flow

Here's the full computational graph of one forward pass:

```
tokens  →  Embedding Lookup (W_E)  →  residual stream (shape: [seq_len, d_model])
                                            │
           ┌────────────────────────────────┤  × 24 layers
           │                                │
           │    RMSNorm (ln1)               │
           │        ↓                       │
           │    Q = x @ W_Q                 │
           │    K = x @ W_K  + RoPE         │
           │    V = x @ W_V                 │
           │    attn_out = Softmax(QK^T/√d) @ V  │
           │    attn_out = attn_out @ W_O   │
           │        ↓                       │
           │    + ────────────────────────── + (residual connection)
           │                                │
           │    RMSNorm (ln2)               │
           │        ↓                       │
           │    gate = SiLU(x @ W_gate)     │
           │    up   = x @ W_up             │
           │    mlp_out = (gate * up) @ W_down  │
           │        ↓                       │
           └──  + ────────────────────────── + (residual connection)
                                            │
           RMSNorm (ln_final)               │
                ↓                           │
           logits = x @ W_U  →  [seq_len, vocab_size]  →  Softmax → probabilities
```

Every `+` is a residual connection — the original signal passes through unchanged, and each sublayer *adds* its contribution. This is why it's called the "residual stream."

**Next notebook**: [06_forward_pass_trace.ipynb](./06_forward_pass_trace.ipynb) — we'll trace actual numbers through this pipeline.